# Fine-Tune MentalRoBERTa for Mental Health Classification

Fine-tune `mental/mental-roberta-base` on the six-class mental health Reddit dataset: ADHD, Anxiety, Bipolar, Depression, PTSD, and None.

**Inputs**
- `data/processed/train.csv`
- `data/processed/val.csv`
- `data/processed/test.csv`

**Outputs**
- Best checkpoint under `results/models/roberta/best_model`
- Evaluation artifacts under `results/evaluation/roberta`
- Summary JSON with training configuration and test metrics

Use this notebook when Colab GPU access is more practical than running transformer fine-tuning locally.


## 1. Runtime and Dependencies

Use a GPU runtime, then install the transformer, metrics, plotting, and parquet dependencies needed by the notebook.


In [ ]:
!nvidia-smi

In [ ]:
!pip -q install transformers accelerate pandas pyarrow scikit-learn matplotlib seaborn tqdm huggingface_hub

## 2. Mount Drive and Configure Paths

Set `PROJECT_DIR` to the Drive folder that contains this repository. This cell also defines training hyperparameters and label names.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR   = Path('/content/drive/MyDrive/mental_health_fusion')
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'

CHECKPOINT_DIR = PROJECT_DIR / 'results' / 'models' / 'roberta'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

EVAL_OUTPUT_DIR = PROJECT_DIR / 'results' / 'evaluation' / 'roberta'
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ──────────────────────────────────────────────────────────
MODEL_NAME     = 'mental/mental-roberta-base'
MAX_LENGTH     = 512
BATCH_SIZE     = 16      # 16 fits comfortably on a T4; use 32 on A100
LEARNING_RATE  = 2e-5
NUM_EPOCHS     = 5
WEIGHT_DECAY   = 0.01
WARMUP_RATIO   = 0.1    # 10 % of steps used for LR warm-up
GRAD_CLIP      = 1.0
SEED           = 42
NUM_LABELS     = 6

ID_TO_CLASS = {
    0: 'ADHD',
    1: 'Anxiety',
    2: 'Bipolar',
    3: 'Depression',
    4: 'PTSD',
    5: 'None',
}

print('Project dir:', PROJECT_DIR)
print('Checkpoints:', CHECKPOINT_DIR)
print('Eval output:', EVAL_OUTPUT_DIR)

## 3. Imports and Reproducibility

Import the training stack, set all random seeds, and select CUDA when available.


In [ ]:
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from tqdm.auto import tqdm


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 4. Load Processed Splits

Read train, validation, and test CSVs, then print class counts to catch label or split problems early.


In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
val_df   = pd.read_csv(PROCESSED_DIR / 'val.csv')
test_df  = pd.read_csv(PROCESSED_DIR / 'test.csv')

for split_name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    dist = df['class_id'].value_counts().sort_index().rename(ID_TO_CLASS)
    print(f'{split_name} ({len(df)} rows):')
    print(dist.to_string())
    print()

## 5. Dataset and DataLoaders

Tokenize each post on demand and build DataLoaders for train, validation, and test evaluation.


In [ ]:
class MentalHealthDataset(Dataset):
    """Tokenizes text on-the-fly; returns input_ids, attention_mask, and label."""

    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int) -> None:
        self.texts  = df['text'].fillna('').astype(str).tolist()
        self.labels = df['class_id'].tolist()
        self.tok    = tokenizer
        self.maxlen = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        enc = self.tok(
            self.texts[idx],
            max_length=self.maxlen,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = MentalHealthDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset   = MentalHealthDataset(val_df,   tokenizer, MAX_LENGTH)
test_dataset  = MentalHealthDataset(test_df,  tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

## 6. Model, Optimizer, and Scheduler

Load the sequence-classification head, configure AdamW, and create a linear warm-up schedule.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True,   # replaces the pre-trained classification head
)
model = model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters : {total_params:,}')
print(f'Total steps          : {total_steps}')
print(f'Warmup steps         : {warmup_steps}')

## 7. Training and Evaluation Helpers

Define one training epoch and one evaluation pass that return loss, accuracy, macro-F1, predictions, and labels.


In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc='  train', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        total_loss += outputs.loss.item()
        all_preds.extend(outputs.logits.argmax(dim=-1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, f1


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc='  eval ', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_loss += outputs.loss.item()
        all_preds.extend(outputs.logits.argmax(dim=-1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, f1, all_preds, all_labels

## 8. Train and Save the Best Checkpoint

Train for the configured number of epochs and save the checkpoint with the best validation macro-F1.


In [ ]:
history = {
    'train_loss': [], 'train_acc': [], 'train_f1': [],
    'val_loss':   [], 'val_acc':   [], 'val_f1':   [],
}

best_val_f1     = 0.0
best_checkpoint = CHECKPOINT_DIR / 'best_model'

for epoch in range(1, NUM_EPOCHS + 1):
    print(f'\n{"="*60}')
    print(f'Epoch {epoch}/{NUM_EPOCHS}')
    print(f'{"="*60}')

    tr_loss, tr_acc, tr_f1 = train_epoch(model, train_loader, optimizer, scheduler, device)
    va_loss, va_acc, va_f1, _, _ = evaluate(model, val_loader, device)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['train_f1'].append(tr_f1)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)
    history['val_f1'].append(va_f1)

    print(f'  Train  loss={tr_loss:.4f}  acc={tr_acc:.4f}  macro-F1={tr_f1:.4f}')
    print(f'  Val    loss={va_loss:.4f}  acc={va_acc:.4f}  macro-F1={va_f1:.4f}')

    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        model.save_pretrained(str(best_checkpoint))
        tokenizer.save_pretrained(str(best_checkpoint))
        print(f'  -> Best checkpoint saved  (val macro-F1 = {best_val_f1:.4f})')

print(f'\nBest validation macro-F1 : {best_val_f1:.4f}')
print(f'Checkpoint               : {best_checkpoint}')

## 9. Training Curves

Plot loss, accuracy, and macro-F1 over epochs, then save the figure with the evaluation artifacts.


In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric in zip(axes, ['loss', 'acc', 'f1']):
    ax.plot(epochs, history[f'train_{metric}'], marker='o', label='Train')
    ax.plot(epochs, history[f'val_{metric}'],   marker='s', label='Val')
    ax.set_title({'loss': 'Cross-Entropy Loss', 'acc': 'Accuracy', 'f1': 'Macro F1'}[metric])
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(EVAL_OUTPUT_DIR / 'training_curves.png'), dpi=150)
plt.show()

## 10. Test Evaluation

Reload the best checkpoint and evaluate it on the held-out test split.


In [ ]:
best_model = AutoModelForSequenceClassification.from_pretrained(str(best_checkpoint))
best_model = best_model.to(device)

test_loss, test_acc, test_f1, test_preds, test_labels = evaluate(best_model, test_loader, device)

print(f'Test results  (best checkpoint — val macro-F1 = {best_val_f1:.4f})')
print(f'  Loss      : {test_loss:.4f}')
print(f'  Accuracy  : {test_acc:.4f}')
print(f'  Macro-F1  : {test_f1:.4f}')

## 11. Per-Class Report

Save and display precision, recall, F1, and support for each class.


In [ ]:
class_names = [ID_TO_CLASS[i] for i in range(NUM_LABELS)]

report_str  = classification_report(test_labels, test_preds, target_names=class_names, digits=4)
report_dict = classification_report(test_labels, test_preds, target_names=class_names,
                                    digits=4, output_dict=True)

print(report_str)

report_df = pd.DataFrame(report_dict).T
report_df.to_csv(EVAL_OUTPUT_DIR / 'classification_report.csv')
display(report_df.round(4))

## 12. Confusion Matrix

Save both count and row-normalized confusion matrix plots for model diagnosis.


In [ ]:
cm      = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('Confusion Matrix (row-normalized)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(str(EVAL_OUTPUT_DIR / 'confusion_matrix.png'), dpi=150)
plt.show()

## 13. Save Run Summary

Write a compact JSON record with model settings, training history, best validation score, and test metrics.


In [ ]:
summary = {
    'model'             : MODEL_NAME,
    'max_length'        : MAX_LENGTH,
    'batch_size'        : BATCH_SIZE,
    'learning_rate'     : LEARNING_RATE,
    'weight_decay'      : WEIGHT_DECAY,
    'warmup_ratio'      : WARMUP_RATIO,
    'num_epochs'        : NUM_EPOCHS,
    'seed'              : SEED,
    'best_val_macro_f1' : round(best_val_f1, 6),
    'test_accuracy'     : round(test_acc, 6),
    'test_macro_f1'     : round(test_f1, 6),
    'test_loss'         : round(test_loss, 6),
    'checkpoint'        : str(best_checkpoint),
    'history'           : history,
}

summary_path = EVAL_OUTPUT_DIR / 'summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Summary saved to : {summary_path}')
print(f'Model checkpoint : {best_checkpoint}')